In [10]:
import matplotlib.pyplot as plt
import numpy as np
import networkx as nx
import ipywidgets as widgets
from IPython.display import display
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from scipy import linalg

# Function to compute transition matrix for random walk
def get_transition_matrix(G):
    """Get the transition matrix for a random walk on graph G"""
    n = G.number_of_nodes()
    # Map nodes to indices
    node_to_idx = {node: idx for idx, node in enumerate(G.nodes())}
    
    # Create transition matrix
    P = np.zeros((n, n))
    for node in G.nodes():
        i = node_to_idx[node]
        neighbors = list(G.neighbors(node))
        degree = len(neighbors)
        if degree > 0:
            for neighbor in neighbors:
                j = node_to_idx[neighbor]
                P[j, i] = 1.0 / degree
    
    return P, node_to_idx

# Function to evolve probability distribution
def evolve_distribution(P, initial_dist, t):
    """Evolve the probability distribution for t steps"""
    if t == 0:
        return initial_dist
    # Matrix power for t steps
    result = initial_dist.copy()
    for _ in range(t):
        result = P @ result
    return result

# Function to compute stationary distribution
def get_stationary_distribution(G):
    """Compute the stationary distribution (proportional to degree)"""
    degrees = dict(G.degree())
    total_degree = sum(degrees.values())
    node_to_idx = {node: idx for idx, node in enumerate(G.nodes())}
    stationary = np.zeros(len(G.nodes()))
    for node, degree in degrees.items():
        stationary[node_to_idx[node]] = degree / total_degree
    return stationary

# Function to get layout based on graph type
def get_layout(G, graph_name):
    """Choose layout using kamada-kawai layout"""
    pos = nx.kamada_kawai_layout(G)
    return pos

# Function to compute eigenvalues of transition matrix
def get_eigenvalues(P):
    """Compute eigenvalues of transition matrix P"""
    eigenvalues = np.linalg.eigvals(P)
    # Sort by magnitude in descending order
    eigenvalues = eigenvalues[np.argsort(np.abs(eigenvalues))[::-1]]
    return np.real(eigenvalues)

def get_spectral_gap(P):
    """Compute spectral gap: 1 - |λ₂|, where λ₁=1 and λ₂ is second largest eigenvalue"""
    eigenvalues = get_eigenvalues(P)
    if len(eigenvalues) >= 2:
        return 1.0 - np.abs(eigenvalues[1])
    return 0.0

# Global storage for plot elements
plot_cache = {
    'fig': None,
    'ax1': None,
    'ax3': None,
    'ax_spectrum': None,
    'nodes1': None,
    'title1': None,
    'labels1': None,
    'stats_text': None,
    'current_graph': None,
    'current_start': None,
    'P': None,
    'node_to_idx': None,
    'idx_to_node': None,
    'stationary': None,
    'pos': None,
    'tv_line': None,
    'tv_point': None,
    'tv_distances': None,
    'max_time': None,
    'mixing_time': None,
    'mixing_text': None,
    'threshold_line': None,
    'mixing_vline': None,
    'spectrum_bars': None,
    'spectral_gap_rect': None,
    'spectral_gap_text': None
}

def compute_tv_distances(P, initial_dist, stationary, max_time):
    """Precompute TV distances for all time steps"""
    tv_distances = []
    current_dist = initial_dist.copy()
    
    for t in range(max_time + 1):
        tv_dist = 0.5 * np.sum(np.abs(current_dist - stationary))
        tv_distances.append(tv_dist)
        if t < max_time:
            current_dist = P @ current_dist
    
    return np.array(tv_distances)

def find_mixing_time(tv_distances, threshold=0.1):
    """Find the first time when TV distance drops below threshold"""
    indices = np.where(tv_distances <= threshold)[0]
    if len(indices) > 0:
        return indices[0]
    return None

def initialize_plot(graph_name, start_node_idx, max_time):
    """Initialize the plot structure (only called when graph or start node changes)"""
    G = graphs[graph_name]
    n = G.number_of_nodes()
    
    # Get transition matrix
    P, node_to_idx = get_transition_matrix(G)
    idx_to_node = {idx: node for node, idx in node_to_idx.items()}
    
    # Get stationary distribution
    stationary = get_stationary_distribution(G)
    
    # Create initial distribution
    initial_dist = np.zeros(n)
    initial_dist[start_node_idx] = 1.0
    
    # Precompute TV distances
    tv_distances = compute_tv_distances(P, initial_dist, stationary, max_time)
    mixing_time = find_mixing_time(tv_distances, threshold=0.1)
    
    # Store in cache
    plot_cache['P'] = P
    plot_cache['node_to_idx'] = node_to_idx
    plot_cache['idx_to_node'] = idx_to_node
    plot_cache['stationary'] = stationary
    plot_cache['current_graph'] = graph_name
    plot_cache['current_start'] = start_node_idx
    plot_cache['tv_distances'] = tv_distances
    plot_cache['max_time'] = max_time
    plot_cache['mixing_time'] = mixing_time
    
    # Close old figure if exists
    if plot_cache['fig'] is not None:
        plt.close(plot_cache['fig'])
    
    # Create figure with subplots: two on top, colorbar in middle, spectrum at bottom
    fig = plt.figure(figsize=(20, 16))
    gs = fig.add_gridspec(3, 2, height_ratios=[8, 0.5, 4], width_ratios=[1, 1], hspace=0.4, wspace=0.3)
    ax1 = fig.add_subplot(gs[0, 0])
    ax3 = fig.add_subplot(gs[0, 1])
    cax = fig.add_subplot(gs[1, :])
    ax_spectrum = fig.add_subplot(gs[2, :])
    
    plot_cache['fig'] = fig
    plot_cache['ax1'] = ax1
    plot_cache['ax3'] = ax3
    plot_cache['ax_spectrum'] = ax_spectrum
    
    # Get layout
    pos = get_layout(G, graph_name)
    plot_cache['pos'] = pos
    
    # Determine node sizes
    if n <= 10:
        node_size = 500
        font_size = 10
    elif n <= 20:
        node_size = 300
        font_size = 8
    elif n <= 40:
        node_size = 200
        font_size = 6
    else:
        node_size = 100
        font_size = 0
    
    # Color map
    cmap = plt.cm.RdYlBu_r
    norm = Normalize(vmin=0, vmax=max(stationary.max() * 1.1, 0.1))
    
    # Plot 1: Current distribution
    plot_cache['title1'] = ax1.set_title(f'Time step: 0\nCurrent Distribution', 
                                         fontsize=12, fontweight='bold')
    
    # Draw edges (static)
    nx.draw_networkx_edges(G, pos, ax=ax1, width=1.5, alpha=0.3, edge_color='gray')
    
    # Draw nodes (will be updated)
    node_colors = [initial_dist[node_to_idx[node]] for node in G.nodes()]
    
    nodes1 = nx.draw_networkx_nodes(G, pos, ax=ax1, node_color=node_colors, 
                                    node_size=node_size, cmap=cmap, vmin=0, 
                                    vmax=norm.vmax, edgecolors='black', linewidths=1)
    plot_cache['nodes1'] = nodes1
    
    if font_size > 0:
        labels1 = nx.draw_networkx_labels(G, pos, ax=ax1, font_size=font_size)
        plot_cache['labels1'] = labels1
    
    ax1.axis('off')
    
    # Plot 2: TV distance over time
    ax3.set_title('Total Variation Distance to Stationary Distribution', 
                  fontsize=12, fontweight='bold')
    ax3.set_xlabel('Time Step', fontsize=11)
    ax3.set_ylabel('TV Distance', fontsize=11)
    ax3.grid(True, alpha=0.3)
    
    # Plot full TV distance curve
    times = np.arange(len(tv_distances))
    tv_line, = ax3.plot(times, tv_distances, 'b-', linewidth=2, label='TV Distance')
    plot_cache['tv_line'] = tv_line
    
    # Add threshold line
    threshold_line = ax3.axhline(y=0.1, color='r', linestyle='--', linewidth=2, 
                                  label='Threshold (ε = 0.1)')
    plot_cache['threshold_line'] = threshold_line
    
    # Add current time point marker
    tv_point, = ax3.plot([0], [tv_distances[0]], 'ro', markersize=10, 
                         label='Current time')
    plot_cache['tv_point'] = tv_point
    
    # Add mixing time annotation
    if mixing_time is not None:
        mixing_text = ax3.text(0.02, 0.98, 
                              f'Mixing time (ε = 0.1): t = {mixing_time}',
                              transform=ax3.transAxes,
                              fontsize=11, fontweight='bold',
                              verticalalignment='top',
                              bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        plot_cache['mixing_text'] = mixing_text
        
        # Add vertical line at mixing time
        mixing_vline = ax3.axvline(x=mixing_time, color='g', linestyle=':', linewidth=2, alpha=0.7)
        plot_cache['mixing_vline'] = mixing_vline
    else:
        mixing_text = ax3.text(0.02, 0.98, 
                              f'Mixing time (ε = 0.1): > {max_time}',
                              transform=ax3.transAxes,
                              fontsize=11, fontweight='bold',
                              verticalalignment='top',
                              bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        plot_cache['mixing_text'] = mixing_text
        plot_cache['mixing_vline'] = None
    
    ax3.legend(loc='upper right', fontsize=10)
    ax3.set_xlim(-1, max_time + 1)
    ax3.set_ylim(-0.05, min(1.05, tv_distances.max() * 1.1))
    
    # Add colorbar (static)
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cax, orientation='horizontal')
    cbar.set_label('Probability', fontsize=10)
    
    # Plot 3: Spectrum of transition matrix
    eigenvalues = get_eigenvalues(P)
    spectral_gap = get_spectral_gap(P)
    
    ax_spectrum.set_title('Spectrum of Transition Matrix', fontsize=12, fontweight='bold')
    ax_spectrum.set_xlabel('Eigenvalue Index', fontsize=11)
    ax_spectrum.set_ylabel('Eigenvalue Magnitude |λ|', fontsize=11)
    ax_spectrum.grid(True, alpha=0.3, axis='y')
    
    # Plot eigenvalues as bars
    indices = np.arange(len(eigenvalues))
    colors = ['red' if i == 0 else 'blue' for i in range(len(eigenvalues))]
    bars = ax_spectrum.bar(indices, np.abs(eigenvalues), color=colors, alpha=0.7, edgecolor='black')
    plot_cache['spectrum_bars'] = bars
    
    # Highlight spectral gap with a shaded rectangle
    if len(eigenvalues) >= 2:
        lambda_1 = np.abs(eigenvalues[0])  # Should be 1.0
        lambda_2 = np.abs(eigenvalues[1])
        gap_rect = ax_spectrum.add_patch(plt.Rectangle((0.3, lambda_2), 0.4, lambda_1 - lambda_2,
                                                       alpha=0.3, color='green', label='Spectral Gap'))
        plot_cache['spectral_gap_rect'] = gap_rect
        
        # Add text annotation for spectral gap
        gap_text = ax_spectrum.text(0.02, 0.95, 
                                   f'Spectral Gap = 1 - |λ₂| = {spectral_gap:.6f}',
                                   transform=ax_spectrum.transAxes,
                                   fontsize=11, fontweight='bold',
                                   verticalalignment='top',
                                   bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
        plot_cache['spectral_gap_text'] = gap_text
    
    ax_spectrum.set_xlim(-0.5, len(eigenvalues) - 0.5)
    ax_spectrum.set_ylim(0, 1.05)
    ax_spectrum.set_xticks(indices[::max(1, len(eigenvalues)//10)])  # Show subset of ticks
    ax_spectrum.legend(loc='upper right', fontsize=10)
    
    # Add text for statistics
    plot_cache['stats_text'] = fig.text(0.5, -0.01, '', ha='center', fontsize=9, 
                                        family='monospace')

def update_plot(time_step):
    """Update only the node colors and title (fast)"""
    if plot_cache['P'] is None:
        return
    
    G = graphs[plot_cache['current_graph']]
    P = plot_cache['P']
    node_to_idx = plot_cache['node_to_idx']
    idx_to_node = plot_cache['idx_to_node']
    stationary = plot_cache['stationary']
    start_node_idx = plot_cache['current_start']
    tv_distances = plot_cache['tv_distances']
    n = len(G.nodes())
    
    # Create initial distribution
    initial_dist = np.zeros(n)
    initial_dist[start_node_idx] = 1.0
    
    # Evolve distribution
    current_dist = evolve_distribution(P, initial_dist, time_step)
    
    # Update node colors
    node_colors = np.array([current_dist[node_to_idx[node]] for node in G.nodes()])
    plot_cache['nodes1'].set_array(node_colors)
    
    # Update title
    plot_cache['title1'].set_text(f'Time step: {time_step}\nCurrent Distribution')
    
    # Update TV distance point
    if time_step < len(tv_distances):
        plot_cache['tv_point'].set_data([time_step], [tv_distances[time_step]])
    
    # Update statistics
    total_variation = tv_distances[time_step] if time_step < len(tv_distances) else 0
    l2_distance = np.linalg.norm(current_dist - stationary)
    
    mixing_time_str = f"{plot_cache['mixing_time']}" if plot_cache['mixing_time'] is not None else f"> {plot_cache['max_time']}"
    
    stats_str = (f"Starting node: {idx_to_node[start_node_idx]}  |  "
                f"Mixing time (ε=0.1): {mixing_time_str}  |  "
                f"Current TV distance: {total_variation:.6f}  |  "
                f"L2 distance: {l2_distance:.6f}  |  "
                f"Max prob: {current_dist.max():.6f}")
    plot_cache['stats_text'].set_text(stats_str)
    
    # Redraw only what changed
    plot_cache['fig'].canvas.draw_idle()

# Visualization function
def visualize_random_walk(graph_name, start_node_idx, time_step):
    # Get current max time from slider
    current_max_time = time_slider.max
    
    # Check if we need to reinitialize the figure
    if (plot_cache['current_graph'] != graph_name or 
        plot_cache['max_time'] != current_max_time or
        plot_cache['fig'] is None):
        initialize_plot(graph_name, start_node_idx, current_max_time)
    elif plot_cache['current_start'] != start_node_idx:
        # Starting node changed - recompute data but keep figure structure
        G = graphs[graph_name]
        P, node_to_idx = get_transition_matrix(G)
        idx_to_node = {idx: node for node, idx in node_to_idx.items()}
        stationary = get_stationary_distribution(G)
        n = G.number_of_nodes()
        
        initial_dist = np.zeros(n)
        initial_dist[start_node_idx] = 1.0
        
        tv_distances = compute_tv_distances(P, initial_dist, stationary, plot_cache['max_time'])
        mixing_time = find_mixing_time(tv_distances, threshold=0.1)
        
        plot_cache['P'] = P
        plot_cache['node_to_idx'] = node_to_idx
        plot_cache['idx_to_node'] = idx_to_node
        plot_cache['stationary'] = stationary
        plot_cache['current_start'] = start_node_idx
        plot_cache['tv_distances'] = tv_distances
        plot_cache['mixing_time'] = mixing_time
        
        # Update TV distance curve
        ax3 = plot_cache['ax3']
        times = np.arange(len(tv_distances))
        plot_cache['tv_line'].set_data(times, tv_distances)
        
        # Update y-axis limits for TV distance
        ax3.set_ylim(-0.05, min(1.05, tv_distances.max() * 1.1))
        
        # Update mixing time annotation
        if mixing_time is not None:
            new_text = f'Mixing time (ε = 0.1): t = {mixing_time}'
            if plot_cache['mixing_vline'] is None:
                plot_cache['mixing_vline'] = ax3.axvline(x=mixing_time, color='g', linestyle=':', linewidth=2, alpha=0.7)
            else:
                plot_cache['mixing_vline'].set_xdata([mixing_time])
        else:
            new_text = f'Mixing time (ε = 0.1): > {plot_cache["max_time"]}'
            if plot_cache['mixing_vline'] is not None:
                plot_cache['mixing_vline'].remove()
                plot_cache['mixing_vline'] = None
        
        plot_cache['mixing_text'].set_text(new_text)
    
    # Update for current time step
    update_plot(time_step)
    
    # Display the figure
    display(plot_cache['fig'])

# Create sample graphs
def create_sample_graphs():
    graphs = {}
    
    # 1. Simple Loop (Cycle)
    graphs['Loop (n=21)'] = nx.cycle_graph(21)
    
    # 2. Tree (Binary tree)
    graphs['Binary Tree (depth=4)'] = nx.balanced_tree(2, 4)
    
    # 3. Complete Graph (Densely connected)
    graphs['Complete Graph (n=15)'] = nx.complete_graph(15)
    
    # 4. Barbell Graph (Graph with bottleneck)
    graphs['Barbell (bottleneck)'] = nx.barbell_graph(8, 1)
    
    # 4b. Three Dense Regions with Two Bottlenecks
    # Create three cliques connected by single-edge bottlenecks
    G_tripartite = nx.Graph()
    size = 9
    # Add three complete graphs (cliques)
    clique1 = nx.complete_graph(size)
    clique2 = nx.complete_graph(size)
    clique3 = nx.complete_graph(size)
    # Relabel nodes to avoid conflicts
    clique1 = nx.relabel_nodes(clique1, {i: i for i in range(size)})
    clique2 = nx.relabel_nodes(clique2, {i: size + i for i in range(size)})
    clique3 = nx.relabel_nodes(clique3, {i: 2*size + i for i in range(size)})
    # Combine and add bottleneck edges
    G_tripartite = nx.compose_all([clique1, clique2, clique3])
    G_tripartite.add_edge(size - 1, size)  # Bottleneck 1
    G_tripartite.add_edge(2*size - 1, 2*size)  # Bottleneck 2
    graphs['3-region (2 bottlenecks)'] = G_tripartite
    
    # 4c. Four Dense Regions with Three Bottlenecks
    # Create four cliques connected by single-edge bottlenecks
    size = 9
    clique1 = nx.complete_graph(size)
    clique2 = nx.complete_graph(size)
    clique3 = nx.complete_graph(size)
    clique4 = nx.complete_graph(size)
    # Relabel nodes to avoid conflicts
    clique1 = nx.relabel_nodes(clique1, {i: i for i in range(size)})
    clique2 = nx.relabel_nodes(clique2, {i: size + i for i in range(size)})
    clique3 = nx.relabel_nodes(clique3, {i: 2*size + i for i in range(size)})
    clique4 = nx.relabel_nodes(clique4, {i: 3*size + i for i in range(size)})
    # Combine and add bottleneck edges
    G_quad = nx.compose_all([clique1, clique2, clique3, clique4])
    G_quad.add_edge(size - 1, size)  # Bottleneck 1
    G_quad.add_edge(2*size - 1, 2*size)  # Bottleneck 2
    G_quad.add_edge(3*size - 1, 3*size)  # Bottleneck 3
    graphs['4-region (3 bottlenecks)'] = G_quad
    
    # 4d. Three Dense Regions with Three Bottlenecks (cyclic)
    # Create three cliques connected cyclically
    size = 9
    clique1 = nx.complete_graph(size)
    clique2 = nx.complete_graph(size)
    clique3 = nx.complete_graph(size)
    # Relabel nodes to avoid conflicts
    clique1 = nx.relabel_nodes(clique1, {i: i for i in range(size)})
    clique2 = nx.relabel_nodes(clique2, {i: size + i for i in range(size)})
    clique3 = nx.relabel_nodes(clique3, {i: 2*size + i for i in range(size)})
    # Combine and add bottleneck edges in a cycle
    G_cycle = nx.compose_all([clique1, clique2, clique3])
    G_cycle.add_edge(size - 1, size)  # Bottleneck 1: Region 1 to Region 2
    G_cycle.add_edge(2*size - 1, 2*size)  # Bottleneck 2: Region 2 to Region 3
    G_cycle.add_edge(3*size - 1, 0)  # Bottleneck 3: Region 3 back to Region 1
    graphs['3-region (3 bottlenecks, cyclic)'] = G_cycle
    
    # 5. Square Grid
    graphs['Grid 5x5'] = nx.grid_2d_graph(5, 5)
    
    # 6. Path (simple chain)
    graphs['Path (n=31)'] = nx.path_graph(31)
    
    # 7. Star Graph (one central node)
    graphs['Star (n=15)'] = nx.star_graph(14)
    
    # 8. Lollipop (tree + cycle)
    graphs['Lollipop'] = nx.lollipop_graph(8, 8)
    
    return graphs

# Create the graphs
graphs = create_sample_graphs()

# Create widgets
graph_dropdown_rw = widgets.Dropdown(
    options=sorted(list(graphs.keys())),
    value='Barbell (bottleneck)',
    description='Graph:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Function to update start node options based on selected graph
def update_start_node_options(change=None):
    G = graphs[graph_dropdown_rw.value]
    n = G.number_of_nodes()
    start_node_slider.max = n - 1
    start_node_slider.value = 0
    time_slider.max = 200
    time_slider.value = 0
    play_widget.max = 200

graph_dropdown_rw.observe(update_start_node_options, 'value')

start_node_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=10,
    step=1,
    description='Start Node:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

time_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=200,
    step=1,
    description='Time Step:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px'),
    continuous_update=True
)

# Play animation widget
play_widget = widgets.Play(
    value=0,
    min=0,
    max=200,
    step=1,
    interval=200,
    description="Play",
    disabled=False
)

# Link play widget to time slider
widgets.jslink((play_widget, 'value'), (time_slider, 'value'))

# Initialize sliders with correct values
update_start_node_options()

# Create output
output_rw = widgets.interactive_output(
    visualize_random_walk, 
    {
        'graph_name': graph_dropdown_rw,
        'start_node_idx': start_node_slider,
        'time_step': time_slider
    }
)

# Display widgets
controls = widgets.VBox([
    graph_dropdown_rw,
    start_node_slider,
    widgets.HBox([play_widget, time_slider])
])

display(controls, output_rw)# Create output


Output()